# PMD Parser / Per-Unit Comparison — Walkthrough

This notebook mirrors `test_pmd_parser_pu.jl` and walks through each validation step for the **IEEE 37-bus** and **IEEE 123-bus** feeders.

The test cross-validates two independent parsers on the same `.dss` files:
- **FeederFlow** — our custom parser
- **PowerModelsDistribution (PMD)** — the reference library

Three things are checked per feeder:
1. **Phase connections** — do `from`/`to` phases agree?
2. **Branch impedance (Z = R + jX)** — in per-unit, to 1e-8 relative tolerance
3. **Load power (P + jQ)** — in per-unit, and wye/delta connection type

## Step 1 — Activate project and load packages

In [1]:
import Pkg
# Activate the FeederFlow project one level up from this notebook
Pkg.activate(joinpath(@__DIR__, ".."))
    
using FeederFlow
using PowerModelsDistribution
using LinearAlgebra
using Printf

println("FeederFlow loaded ✓")
println("PowerModelsDistribution loaded ✓")

FeederFlow loaded ✓

  Activating project at `c:\Users\hoang\OneDrive - Massachusetts Institute of Technology\1. MIT\2. Projects\4. Dist OPF\multiphase_modelling\FeederFlow.jl`



PowerModelsDistribution loaded ✓


## Step 2 — File paths

In [2]:
REPO_ROOT  = normpath(joinpath(@__DIR__, "..", ".."))
IEEE37_DSS = joinpath(REPO_ROOT, "three-phase-modeling", "IEEE 37-bus feeder",
                      "IEEE37openDSSdata", "ieee37opendss.dss")
IEEE123_DSS = joinpath(REPO_ROOT, "three-phase-modeling", "IEEE 123-bus feeder",
                       "IEEE123openDSSdata", "IEEE123Master.dss")

println("IEEE37  → ", IEEE37_DSS)
println("  exists: ", isfile(IEEE37_DSS))
println("IEEE123 → ", IEEE123_DSS)
println("  exists: ", isfile(IEEE123_DSS))

IEEE37  → c:\Users\hoang\OneDrive - Massachusetts Institute of Technology\1. MIT\2. Projects\4. Dist OPF\multiphase_modelling\three-phase-modeling\IEEE 37-bus feeder\IEEE37openDSSdata\ieee37opendss.dss
  exists: true
IEEE123 → c:\Users\hoang\OneDrive - Massachusetts Institute of Technology\1. MIT\2. Projects\4. Dist OPF\multiphase_modelling\three-phase-modeling\IEEE 123-bus feeder\IEEE123openDSSdata\IEEE123Master.dss
  exists: true


## Step 3 — Helper functions (mirrors the test file)

In [3]:
"""
Parse a DSS file into PMD's MATHEMATICAL data model (per-unit, no Kron reduction).
`make_pu=true` means PMD normalises everything by vbase/sbase before returning.
"""
function pmd_math_model(path)
    PowerModelsDistribution.parse_file(
        path;
        data_model = PowerModelsDistribution.MATHEMATICAL,
        make_pu    = true,
        kron_reduce  = false,
        phase_project = false,
    )
end

"""Build name→branch dict from PMD's numbered branch dict."""
pmd_branch_map(data) =
    Dict(String(b["name"]) => b for b in values(data["branch"]) if haskey(b, "name"))

"""Build name→load dict from PMD's numbered load dict."""
pmd_load_map(data) =
    Dict(String(l["name"]) => l for l in values(data["load"]) if haskey(l, "name"))

"""
Convert a FeederFlow LineDevice to a per-unit complex impedance matrix.

Formula:  Z_pu = Z_Ω · length · (sbase_MVA / Vbase_kV²)
          where sbase comes from PMD in kVA  →  /1000 → MVA
"""
function line_pu_matrix(line::LineDevice, branch, sbase::Float64)
    vbase = Float64(branch["vbase"])          # kV
    scale = (sbase / 1000) / vbase^2          # Ω → pu conversion factor
    complex.(line.rmatrix, line.xmatrix) .* line.length .* scale
end

connection_key(phases) = sort(collect(phases))

connection_config(conn::Symbol) = conn == :delta ? "DELTA" : "WYE"

println("Helpers defined ✓")

Helpers defined ✓


## Step 4 — Parse IEEE 37-bus with both parsers

In [4]:
println("=" ^ 60)
println("IEEE 37-BUS FEEDER")
println("=" ^ 60)

print("Parsing with FeederFlow... ")
net37 = FeederFlow.parse_file(IEEE37_DSS)
println("done")

print("Parsing with PMD...        ")
math37  = pmd_math_model(IEEE37_DSS)
branches37 = pmd_branch_map(math37)
loads37    = pmd_load_map(math37)
sbase37    = Float64(math37["settings"]["sbase"])   # kVA
println("done")

println()
println("─── System base ──────────────────────────────────────")
@printf("  sbase  = %.1f kVA  (= %.1f MVA)\n", sbase37, sbase37/1000)
println()
println("─── FeederFlow counts ────────────────────────────────")
println("  buses  : ", length(net37.buses))
println("  lines  : ", length(net37.lines))
println("  loads  : ", length(net37.loads))
println()
println("─── PMD counts ───────────────────────────────────────")
println("  branches (named): ", length(branches37))
println("  loads    (named): ", length(loads37))

IEEE 37-BUS FEEDER
Parsing with FeederFlow... done
Parsing with PMD...        

[ PowerModelsDistribution | Info ] : Circuit has been reset with the 'clear' on line 1 in 'ieee37opendss.dss'
[ PowerModelsDistribution | Info ] : Redirecting to 'ieeelinecodes.dss' on line 21 in 'ieee37opendss.dss'
[ PowerModelsDistribution | Info ] : Command 'calcvoltagebases' on line 104 in 'ieee37opendss.dss' is not supported, skipping.
[ PowerModelsDistribution | Info ] : Reading Buscoords in 'ieee37_busxy.csv' on line 105 in 'IEEELineCodes.DSS'
[ PowerModelsDistribution | Info ] : Command 'solve' on line 109 in 'ieee37opendss.dss' is not supported, skipping.
┌ PowerModelsDistribution | Warning ] : DssOptions has no field `maxiterations`, skipping...
└ @ PowerModelsDistribution C:\Users\hoang\.julia\packages\PowerModelsDistribution\EOTxp\src\data_model\transformations\rawdss2dss.jl:45
┌ PowerModelsDistribution | Warning ] : dss load model 4 not supported; treating as constant POWER model
└ @ PowerModelsDistribution C:\Users\hoang\.julia\packages\PowerModelsDistribution\EOTxp\src\d

done

─── System base ──────────────────────────────────────
  sbase  = 100000.0 kVA  (= 100.0 MVA)

─── FeederFlow counts ────────────────────────────────
  buses  : 39
  lines  : 36
  loads  : 30

─── PMD counts ───────────────────────────────────────
  branches (named): 49
  loads    (named): 30


In [5]:
net37.lines[2].provenance


Provenance("ieee37opendss.dss", "line.jumper", Dict{String, Any}("c1" => 0.0, "phases" => 1.0, "bus1" => 799.2, "r1" => 0.001, "x1" => 0.0, "x0" => 0.0, "r0" => 0.001, "c0" => 0.0, "bus2" => "799r.2"), "New Line.Jumper Phases=1 Bus1=799.2      Bus2=799r.2     r0=1e-3 r1=1e-3 x0=0 x1=0 c0=0 c1=0")

In [6]:
math37["branch"]

Dict{String, Any} with 49 entries:
  "32" => Dict{String, Any}("br_r"=>[2.68679 0.667337 0.631688; 0.667337 2.7016…
  "29" => Dict{String, Any}("br_r"=>[1.91406 0.720733 0.678415; 0.720733 1.9267…
  "1"  => Dict{String, Any}("br_r"=>[0.893229 0.336342 0.316594; 0.336342 0.899…
  "2"  => Dict{String, Any}("br_r"=>[0.638021 0.240244 0.226138; 0.240244 0.642…
  "41" => Dict{String, Any}("br_r"=>[0.0 0.0 0.0; 0.0 0.0 0.0; 0.0 0.0 0.0], "b…
  "27" => Dict{String, Any}("br_r"=>[1.12476 0.385653 0.29214; 0.385653 1.0625 …
  "42" => Dict{String, Any}("br_r"=>[1.2 0.0 0.0; 0.0 1.2 0.0; 0.0 0.0 1.2], "b…
  "33" => Dict{String, Any}("br_r"=>[1.44673 0.359336 0.34014; 0.359336 1.45474…
  "28" => Dict{String, Any}("br_r"=>[1.33491 0.307038 0.153747; 0.307038 1.2071…
  "26" => Dict{String, Any}("br_r"=>[1.02083 0.384391 0.361821; 0.384391 1.0276…
  "10" => Dict{String, Any}("br_r"=>[1.24006 0.308002 0.291548; 0.308002 1.2469…
  "24" => Dict{String, Any}("br_r"=>[1.03338 0.256668 0.242957; 0.256668 1

In [7]:
net37.lines

36-element ComponentTable{LineDevice}:
 LineDevice("l12", TerminalSpec("707", [1, 2, 3]), TerminalSpec("724", [1, 2, 3]), [1, 2, 3], "724", 0.76, [0.396818182 0.098560606 0.093295455; 0.098560606 0.399015152 0.098560606; 0.093295455 0.098560606 0.396818182], [0.146931818 0.051856061 0.040208333; 0.051856061 0.140113636 0.051856061; 0.040208333 0.051856061 0.146931818], [30.26701029 0.0 0.0; 0.0 30.26701029 0.0; 0.0 0.0 30.26701029], "none", 60.0, Provenance("ieee37opendss.dss", "line.l12", Dict{String, Any}("length" => 0.76, "phases" => 3.0, "bus1" => "707.1.2.3", "linecode" => 724.0, "bus2" => "724.1.2.3"), "New Line.L12    Phases=3 Bus1=707.1.2.3  Bus2=724.1.2.3  LineCode=724  Length=0.76"), false, true)
 LineDevice("jumper", TerminalSpec("799", [2]), TerminalSpec("799r", [2]), [2], nothing, 1.0, [0.001;;], [0.0;;], [0.0;;], "none", 60.0, Provenance("ieee37opendss.dss", "line.jumper", Dict{String, Any}("c1" => 0.0, "phases" => 1.0, "bus1" => 799.2, "r1" => 0.001, "x1" => 0.0, "x0" =>

In [8]:
net37.lines[1].provenance

Provenance("ieee37opendss.dss", "line.l12", Dict{String, Any}("length" => 0.76, "phases" => 3.0, "bus1" => "707.1.2.3", "linecode" => 724.0, "bus2" => "724.1.2.3"), "New Line.L12    Phases=3 Bus1=707.1.2.3  Bus2=724.1.2.3  LineCode=724  Length=0.76")

In [9]:
Ymatrix = build_y(net37)

YBusModel(sparse([1, 2, 3, 4, 5, 6, 13, 14, 15, 64  …  9, 115, 116, 117, 7, 8, 9, 115, 116, 117], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1  …  116, 116, 116, 116, 117, 117, 117, 117, 117, 117], ComplexF64[81.6468056459581 - 28.472026092630685im, -18.230370945579782 + 3.4143928094662437im, -15.058627544367114 + 5.531132007407874im, -10.006446873128079 + 3.489470652160008im, 2.2342728157104887 - 0.4184602199878699im, 1.8455511555263424 - 0.6778829636077144im, -8.266195243018847 + 2.8826061909147893im, 1.8457036303695333 - 0.34568452955519685im, 1.5245857371739346 - 0.5599902742846338im, -63.37416352981117 + 22.099980797013387im  …  0.006820119352088664 - 0.027280477408354657im, -0.0001423329256088069 + 0.0005693317024352276im, 0.0002846658552598742 - 0.0011386634048704551im, -0.0001423329256088069 + 0.0005693317024352276im, 0.006820119352088664 - 0.027280477408354657im, 0.006820119352088664 - 0.027280477408354657im, -0.013640238704177328 + 0.05456095481670931im, -0.0001423329256088069 + 0.000569331

In [10]:
Ymatrix.network_order[1]

BusPhase("707", 1)

## Step 5 — IEEE 37: Branch (line) comparison

For each FeederFlow line that also exists in PMD's branch map we check:
- Phase connection arrays match
- `br_r` and `br_x` (per-unit) match our own scaling to 1e-8 relative tolerance

In [11]:
println("Branch comparison — IEEE 37")
println("-" ^ 60)
@printf("  %-30s  %6s  %6s  %12s  %12s\n",
        "line", "phases", "match", "max_ΔR_pu", "max_ΔX_pu")
println("-" ^ 60)

pass_lines = 0; fail_lines = 0; skip_lines = 0

for line in sort(net37.lines, by = l -> l.name)
    if !haskey(branches37, line.name)
        skip_lines += 1
        continue
    end
    branch = branches37[line.name]

    # Phase connection check
    from_ok = sort(branch["f_connections"]) == sort(collect(line.from.phases))
    to_ok   = sort(branch["t_connections"]) == sort(collect(line.to.phases))
    phases_ok = from_ok && to_ok

    # Impedance check
    Z_pu   = line_pu_matrix(line, branch, sbase37)
    ΔR     = maximum(abs.(branch["br_r"] .- real(Z_pu)))
    ΔX     = maximum(abs.(branch["br_x"] .- imag(Z_pu)))
    imp_ok = isapprox(branch["br_r"], real(Z_pu); rtol=1e-8, atol=1e-10) &&
             isapprox(branch["br_x"], imag(Z_pu); rtol=1e-8, atol=1e-10)

    ok = phases_ok && imp_ok
    ok ? (pass_lines += 1) : (fail_lines += 1)

    status = ok ? "✓" : "✗ FAIL"
    @printf("  %-30s  %-6s  %-6s  %12.2e  %12.2e\n",
            line.name, string(sort(collect(line.from.phases))), status, ΔR, ΔX)
end

println("-" ^ 60)
@printf("  Passed: %d  |  Failed: %d  |  Skipped (no PMD match): %d\n",
        pass_lines, fail_lines, skip_lines)

Branch comparison — IEEE 37
------------------------------------------------------------
  line                            phases   match     max_ΔR_pu     max_ΔX_pu
------------------------------------------------------------
  jumper                          [2]     ✓           1.73e-18      0.00e+00
  l1                              [1, 2, 3]  ✓           5.55e-17      1.11e-16
  l10                             [1, 2, 3]  ✓           2.22e-16      5.55e-17
  l11                             [1, 2, 3]  ✓           2.22e-16      1.11e-16
  l12                             [1, 2, 3]  ✓           4.44e-16      2.22e-16
  l13                             [1, 2, 3]  ✓           1.11e-16      2.78e-17
  l14                             [1, 2, 3]  ✓           2.22e-16      1.11e-16
  l15                             [1, 2, 3]  ✓           2.22e-16      1.11e-16
  l16                             [1, 2, 3]  ✓           4.44e-16      1.11e-16
  l17                             [1, 2, 3]  ✓          

## Step 6 — IEEE 37: Load comparison

For each load we verify:
- Connection array (phases) matches — filtering out neutral node 4
- Node 4 present ↔ `conn == :wye`
- Configuration string matches (`"WYE"` / `"DELTA"`)
- Total P (kw) and Q (kvar) normalised by `sbase` match PMD's `pd` / `qd`

In [12]:
println("Load comparison — IEEE 37")
println("-" ^ 80)
@printf("  %-20s  %-6s  %-6s  %10s  %10s  %10s  %10s  %6s\n",
        "load", "conn", "match", "kw", "pd_pu", "kvar", "qd_pu", "ok")
println("-" ^ 80)

pass_loads = 0; fail_loads = 0

for load in sort(net37.loads, by = l -> l.name)
    ml = loads37[load.name]

    conn_phases = sort(filter(!=(4), ml["connections"]))
    phases_ok   = conn_phases == connection_key(load.bus.phases)
    neutral_ok  = (4 in ml["connections"]) == (load.conn == :wye)
    config_ok   = string(ml["configuration"]) == connection_config(load.conn)
    p_ok        = isapprox(sum(ml["pd"]), load.kw   / sbase37; rtol=1e-8, atol=1e-10)
    q_ok        = isapprox(sum(ml["qd"]), load.kvar / sbase37; rtol=1e-8, atol=1e-10)

    ok = phases_ok && neutral_ok && config_ok && p_ok && q_ok
    ok ? (pass_loads += 1) : (fail_loads += 1)

    @printf("  %-20s  %-6s  %-6s  %10.2f  %10.6f  %10.2f  %10.6f  %6s\n",
            load.name, string(load.conn),
            string(connection_key(load.bus.phases)),
            load.kw,  load.kw   / sbase37,
            load.kvar, load.kvar / sbase37,
            ok ? "✓" : "✗ FAIL")
end

println("-" ^ 80)
@printf("  Passed: %d  |  Failed: %d\n", pass_loads, fail_loads)

Load comparison — IEEE 37
--------------------------------------------------------------------------------
  load                  conn    match           kw       pd_pu        kvar       qd_pu      ok
--------------------------------------------------------------------------------
  s701a                 delta   [1, 2]      140.00    0.001400       70.00    0.000700       ✓
  s701b                 delta   [2, 3]      140.00    0.001400       70.00    0.000700       ✓
  s701c                 delta   [1, 3]      350.00    0.003500      175.00    0.001750       ✓
  s712c                 delta   [1, 3]       85.00    0.000850       40.00    0.000400       ✓
  s713c                 delta   [1, 3]       85.00    0.000850       40.00    0.000400       ✓
  s714a                 delta   [1, 2]       17.00    0.000170        8.00    0.000080       ✓
  s714b                 delta   [2, 3]       21.00    0.000210       10.00    0.000100       ✓
  s718a                 delta   [1, 2]       85.00  

## Step 7 — Parse IEEE 123-bus with both parsers

In [13]:
println("=" ^ 60)
println("IEEE 123-BUS FEEDER")
println("=" ^ 60)

print("Parsing with FeederFlow... ")
net123 = FeederFlow.parse_file(IEEE123_DSS)
println("done")

print("Parsing with PMD...        ")
math123    = pmd_math_model(IEEE123_DSS)
branches123 = pmd_branch_map(math123)
loads123    = pmd_load_map(math123)
sbase123    = Float64(math123["settings"]["sbase"])
println("done")

println()
println("─── System base ──────────────────────────────────────")
@printf("  sbase  = %.1f kVA  (= %.1f MVA)\n", sbase123, sbase123/1000)
println()
println("─── FeederFlow counts ────────────────────────────────")
println("  buses  : ", length(net123.buses))
println("  lines  : ", length(net123.lines))
println("  loads  : ", length(net123.loads))
println()
println("─── PMD counts ───────────────────────────────────────")
println("  branches (named): ", length(branches123))
println("  loads    (named): ", length(loads123))

IEEE 123-BUS FEEDER
Parsing with FeederFlow... done
Parsing with PMD...        

[ PowerModelsDistribution | Info ] : Circuit has been reset with the 'clear' on line 8 in 'IEEE123Master.dss'
[ PowerModelsDistribution | Info ] : Redirecting to 'ieeelinecodes.dss' on line 31 in 'IEEE123Master.dss'
[ PowerModelsDistribution | Info ] : Redirecting to 'ieee123regulators.dss' on line 209 in 'IEEELineCodes.DSS'
[ PowerModelsDistribution | Info ] : Redirecting to 'ieee123loads.dss' on line 213 in 'IEEE123Regulators.DSS'
[ PowerModelsDistribution | Info ] : Command 'calcvoltagebases' on line 224 in 'IEEE123Master.dss' is not supported, skipping.
[ PowerModelsDistribution | Info ] : basemva=100 is the default value, you may want to adjust sbase_default for better convergence


done

─── System base ──────────────────────────────────────
  sbase  = 100000.0 kVA  (= 100.0 MVA)

─── FeederFlow counts ────────────────────────────────
  buses  : 132
  lines  : 126
  loads  : 91

─── PMD counts ───────────────────────────────────────
  branches (named): 150
  loads    (named): 91


## Step 8 — IEEE 123: Branch comparison

In [14]:
println("Branch comparison — IEEE 123")
println("-" ^ 60)
@printf("  %-30s  %6s  %6s  %12s  %12s\n",
        "line", "phases", "match", "max_ΔR_pu", "max_ΔX_pu")
println("-" ^ 60)

pass_lines = 0; fail_lines = 0; skip_lines = 0

for line in sort(net123.lines, by = l -> l.name)
    if !haskey(branches123, line.name)
        skip_lines += 1
        continue
    end
    branch = branches123[line.name]

    from_ok = sort(branch["f_connections"]) == sort(collect(line.from.phases))
    to_ok   = sort(branch["t_connections"]) == sort(collect(line.to.phases))
    phases_ok = from_ok && to_ok

    Z_pu   = line_pu_matrix(line, branch, sbase123)
    ΔR     = maximum(abs.(branch["br_r"] .- real(Z_pu)))
    ΔX     = maximum(abs.(branch["br_x"] .- imag(Z_pu)))
    imp_ok = isapprox(branch["br_r"], real(Z_pu); rtol=1e-8, atol=1e-10) &&
             isapprox(branch["br_x"], imag(Z_pu); rtol=1e-8, atol=1e-10)

    ok = phases_ok && imp_ok
    ok ? (pass_lines += 1) : (fail_lines += 1)

    status = ok ? "✓" : "✗ FAIL"
    @printf("  %-30s  %-6s  %-6s  %12.2e  %12.2e\n",
            line.name, string(sort(collect(line.from.phases))), status, ΔR, ΔX)
end

println("-" ^ 60)
@printf("  Passed: %d  |  Failed: %d  |  Skipped (no PMD match): %d\n",
        pass_lines, fail_lines, skip_lines)

Branch comparison — IEEE 123
------------------------------------------------------------
  line                            phases   match     max_ΔR_pu     max_ΔX_pu
------------------------------------------------------------
  l1                              [2]     ✓           0.00e+00      0.00e+00
  l10                             [1, 2, 3]  ✓           0.00e+00      0.00e+00
  l100                            [3]     ✓           0.00e+00      0.00e+00
  l101                            [1, 2, 3]  ✓           0.00e+00      0.00e+00
  l102                            [3]     ✓           0.00e+00      0.00e+00
  l103                            [3]     ✓           0.00e+00      0.00e+00
  l104                            [2]     ✓           0.00e+00      0.00e+00
  l105                            [1, 2, 3]  ✓           0.00e+00      0.00e+00
  l106                            [2]     ✓           0.00e+00      0.00e+00
  l107                            [1]     ✓           0.00e+00      0.

## Step 9 — IEEE 123: Load comparison

In [15]:
println("Load comparison — IEEE 123")
println("-" ^ 80)
@printf("  %-20s  %-6s  %-6s  %10s  %10s  %10s  %10s  %6s\n",
        "load", "conn", "phases", "kw", "pd_pu", "kvar", "qd_pu", "ok")
println("-" ^ 80)

pass_loads = 0; fail_loads = 0

for load in sort(net123.loads, by = l -> l.name)
    ml = loads123[load.name]

    conn_phases = sort(filter(!=(4), ml["connections"]))
    phases_ok   = conn_phases == connection_key(load.bus.phases)
    neutral_ok  = (4 in ml["connections"]) == (load.conn == :wye)
    config_ok   = string(ml["configuration"]) == connection_config(load.conn)
    p_ok        = isapprox(sum(ml["pd"]), load.kw   / sbase123; rtol=1e-8, atol=1e-10)
    q_ok        = isapprox(sum(ml["qd"]), load.kvar / sbase123; rtol=1e-8, atol=1e-10)

    ok = phases_ok && neutral_ok && config_ok && p_ok && q_ok
    ok ? (pass_loads += 1) : (fail_loads += 1)

    @printf("  %-20s  %-6s  %-6s  %10.2f  %10.6f  %10.2f  %10.6f  %6s\n",
            load.name, string(load.conn),
            string(connection_key(load.bus.phases)),
            load.kw,  load.kw   / sbase123,
            load.kvar, load.kvar / sbase123,
            ok ? "✓" : "✗ FAIL")
end

println("-" ^ 80)
@printf("  Passed: %d  |  Failed: %d\n", pass_loads, fail_loads)

Load comparison — IEEE 123
--------------------------------------------------------------------------------
  load                  conn    phases          kw       pd_pu        kvar       qd_pu      ok
--------------------------------------------------------------------------------
  s100c                 wye     [3]          40.00    0.000400       20.00    0.000200       ✓
  s102c                 wye     [3]          20.00    0.000200       10.00    0.000100       ✓
  s103c                 wye     [3]          40.00    0.000400       20.00    0.000200       ✓
  s104c                 wye     [3]          40.00    0.000400       20.00    0.000200       ✓
  s106b                 wye     [2]          40.00    0.000400       20.00    0.000200       ✓
  s107b                 wye     [2]          40.00    0.000400       20.00    0.000200       ✓
  s109a                 wye     [1]          40.00    0.000400       20.00    0.000200       ✓
  s10a                  wye     [1]          20.00 

## Step 10 — Overall summary

In [16]:
println()
println("=" ^ 60)
println("OVERALL SUMMARY")
println("=" ^ 60)

# Recount totals across both feeders
function count_all(net, branches, loads, sbase)
    lp = lf = ls = lop = lof = 0
    for line in net.lines
        !haskey(branches, line.name) && (ls += 1; continue)
        br = branches[line.name]
        Z  = line_pu_matrix(line, br, sbase)
        ok = sort(br["f_connections"]) == sort(collect(line.from.phases)) &&
             sort(br["t_connections"]) == sort(collect(line.to.phases))   &&
             isapprox(br["br_r"], real(Z); rtol=1e-8, atol=1e-10)        &&
             isapprox(br["br_x"], imag(Z); rtol=1e-8, atol=1e-10)
        ok ? (lp += 1) : (lf += 1)
    end
    for load in net.loads
        ml = loads[load.name]
        ok = sort(filter(!=(4), ml["connections"])) == connection_key(load.bus.phases) &&
             (4 in ml["connections"]) == (load.conn == :wye)                           &&
             string(ml["configuration"]) == connection_config(load.conn)               &&
             isapprox(sum(ml["pd"]), load.kw   / sbase; rtol=1e-8, atol=1e-10)        &&
             isapprox(sum(ml["qd"]), load.kvar / sbase; rtol=1e-8, atol=1e-10)
        ok ? (lop += 1) : (lof += 1)
    end
    lp, lf, ls, lop, lof
end

lp37, lf37, ls37, lop37, lof37     = count_all(net37,  branches37,  loads37,  sbase37)
lp123, lf123, ls123, lop123, lof123 = count_all(net123, branches123, loads123, sbase123)

println()
@printf("  %-10s  %8s  %8s  %8s  %8s  %8s\n",
        "feeder", "line✓", "line✗", "skip", "load✓", "load✗")
println("  " * "-" ^ 54)
@printf("  %-10s  %8d  %8d  %8d  %8d  %8d\n",
        "IEEE 37",  lp37,  lf37,  ls37,  lop37,  lof37)
@printf("  %-10s  %8d  %8d  %8d  %8d  %8d\n",
        "IEEE 123", lp123, lf123, ls123, lop123, lof123)
println()

total_fail = lf37 + lf123 + lof37 + lof123
if total_fail == 0
    println("  ALL CHECKS PASSED ✓")
else
    println("  FAILURES DETECTED: ", total_fail, " ✗")
end
println("=" ^ 60)


OVERALL SUMMARY

  feeder         line✓     line✗      skip     load✓     load✗
  ------------------------------------------------------
  IEEE 37           36         0         0        30         0
  IEEE 123         126         0         0        91         0

  ALL CHECKS PASSED ✓


In [ ]:
net123.

LineDevice("l77", TerminalSpec("76", [1, 2, 3]), TerminalSpec("86", [1, 2, 3]), [1, 2, 3], "3", 0.7, [0.087405303 0.02907197 0.029924242; 0.02907197 0.086666667 0.029545455; 0.029924242 0.029545455 0.088371212], [0.201723485 0.072897727 0.080227273; 0.072897727 0.204166667 0.095018939; 0.080227273 0.095018939 0.198522727], [2.71134756 -0.350755566 -0.585011253; -0.350755566 2.851710072 -0.920293787; -0.585011253 -0.920293787 3.004631862], "none", 60.0, Provenance("IEEE123Master.dss", "line.l77", Dict{String, Any}("length" => 0.7, "phases" => 3.0, "bus1" => "76.1.2.3", "linecode" => 3.0, "bus2" => "86.1.2.3"), "New Line.L77    Phases=3 Bus1=76.1.2.3   Bus2=86.1.2.3   LineCode=3    Length=0.7"), false, true)

: 